In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic, FK4, FK5
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
#from scipy.signal import convolve as scipy_convolve
from astropy.convolution import convolve
import gc
from matplotlib.patches import Circle
from scipy.stats import linregress
import matplotlib as mpl
from matplotlib.patches import Ellipse
from radio_beam import Beam
import astropy.units as u
from matplotlib.patches import Rectangle
from reproject import reproject_from_healpix, reproject_to_healpix

In [ ]:
P_thr   = 0.1 # K
dRM_thr = 150 # rad/m^2

In [ ]:
# Read in RM, Pearson R, and standard error in RM:
hdu_RM_CG = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_CG_conv4_regrd.fits')
RM_CG_all = hdu_RM_CG[0].data
RM_CG     = RM_CG_all.copy()
rvalue_CG = hdu_RM_CG[2].data
stderr_CG = hdu_RM_CG[4].data
hdr       = hdu_RM_CG[0].header
wcs = WCS(hdr)

hdu_PA_CG = fits.open('/srv/data/cgps-gmims/conv_regrid/PA_A_C_conv4_regrd.fits')
PA_CG     = hdu_PA_CG[0].data
print(PA_CG.shape)

hdu_RM_G = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_G_regrd.fits')
RM_G_all = hdu_RM_G[0].data
RM_G     = RM_G_all.copy()
rvalue_G = hdu_RM_G[2].data
stderr_G = hdu_RM_G[4].data

hdu_RM_C = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_C_conv4_regrd.fits')
RM_C_all = hdu_RM_C[0].data
RM_C     = RM_C_all.copy()
rvalue_C = hdu_RM_C[2].data
stderr_C = hdu_RM_C[4].data

# Read in polarised intensity:
hdu_PI_CG = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_CG_conv4_regrd_PI_of_mean.fits')
PI_CG     = hdu_PI_CG[0].data

hdu_PI_G = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_G_regrd_PI_of_mean.fits')
PI_G     = hdu_PI_G[0].data 

hdu_PI_C = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_C_conv4_regrd_PI_of_mean.fits')
PI_C     = hdu_PI_C[0].data 

# Set outside of mosaics to NaN:
RM_CG[RM_CG_all==0.0]     = np.nan
rvalue_CG[RM_CG_all==0.0] = np.nan
stderr_CG[RM_CG_all==0.0] = np.nan
PI_CG[RM_CG_all==0.0]     = np.nan

RM_C[RM_C_all==0.0]     = np.nan
rvalue_C[RM_C_all==0.0] = np.nan
stderr_C[RM_C_all==0.0] = np.nan
PI_C[RM_C_all==0.0]     = np.nan

In [ ]:
RM_CG_filt = RM_CG.copy()
RM_G_filt = RM_G.copy()
RM_C_filt = RM_C.copy()
########################################
RM_CG_filt[PI_CG<P_thr] = np.nan
RM_CG_filt[stderr_CG>dRM_thr] = np.nan
RM_G_filt[PI_G<P_thr] = np.nan
RM_G_filt[stderr_G>dRM_thr] = np.nan
#RM_C_filt[PI_C<0.1] = np.nan
RM_C_filt[stderr_C>dRM_thr] = np.nan
########################################

RM = [RM_C_filt,RM_G_filt,RM_CG_filt]
PI = [PI_C,PI_G,PI_CG]

In [ ]:
def make_the_plots_v2(FD_plot, RM_plot, hdrFD, hdrRM, l0_list, b0_list,lonspace_list,latspace_list, 
                      lwidth_list, filename='test',
                       FDmax=50, RMmax=300, src=None):
     
    fs = 24
    fig = plt.figure(figsize=(24.5,10))

    plt.subplots_adjust(hspace=0.05, wspace=0.15, left=0.07, right=0.9, top=0.98, bottom=0.08)

    # PI maps
    
    cmap = mpl.colormaps.get_cmap('RdBu_r') 
    cmap.set_bad(color='grey')
    axFD1  = fig.add_subplot(231, projection=WCS(hdrFD).celestial)
    axFD2  = fig.add_subplot(232, projection=WCS(hdrFD).celestial)
    axFD3  = fig.add_subplot(233, projection=WCS(hdrFD).celestial)
    imFD1  = axFD1.imshow(FD_plot, origin='lower', vmin=-FDmax, vmax=FDmax, cmap=cmap)
    imFD2  = axFD2.imshow(FD_plot, origin='lower', vmin=-FDmax, vmax=FDmax, cmap=cmap)
    imFD3  = axFD3.imshow(FD_plot, origin='lower', vmin=-FDmax, vmax=FDmax, cmap=cmap)

    cb_axFD = fig.add_axes([0.91, 0.545, 0.02, 0.43])
    cbarFD = fig.colorbar(imFD3, cax=cb_axFD, orientation='vertical')
    cbarFD.set_label(r'peak FD (rad m$^{-2}$)', fontsize=fs)

    # RM maps

    axRM1  = fig.add_subplot(234, projection=WCS(hdrRM).celestial)
    axRM2  = fig.add_subplot(235, projection=WCS(hdrRM).celestial)
    axRM3  = fig.add_subplot(236, projection=WCS(hdrRM).celestial)
    imRM1  = axRM1.imshow(RM_plot, origin='lower', vmin=-RMmax, vmax=RMmax, cmap=cmap)
    imRM2  = axRM2.imshow(RM_plot, origin='lower', vmin=-RMmax, vmax=RMmax, cmap=cmap)
    imRM3  = axRM3.imshow(RM_plot, origin='lower', vmin=-RMmax, vmax=RMmax, cmap=cmap)

    cb_axRM = fig.add_axes([0.91, 0.085, 0.02, 0.43])
    cbarRM = fig.colorbar(imRM3, cax=cb_axRM, orientation='vertical')
    cbarRM.set_label(r'RM (rad m$^{-2}$)', fontsize=fs)
    
    fig.text(0.008,0.42,'Galactic Latitude',fontsize=fs,rotation='vertical')
    fig.text(0.42,0.01,'Galactic Longitude',fontsize=fs,rotation='horizontal')

    axsFD = [axFD1,axFD2,axFD3]
    axsRM = [axRM1,axRM2,axRM3]

    for i in range(0,3):

        bwidth = 3*lwidth_list[i]/4
        
        llim=[l0_list[i]+lwidth_list[i]/2,l0_list[i]-lwidth_list[i]/2]
        blim=[b0_list[i]-bwidth/2,        b0_list[i]+bwidth/2]
    
        c = SkyCoord(llim, blim, frame=Galactic, unit="deg")

        axsFD[i].set_xlim(WCS(hdrFD).world_to_pixel(c)[0])
        axsFD[i].set_ylim(WCS(hdrFD).world_to_pixel(c)[1])
        axsRM[i].set_xlim(WCS(hdrRM).world_to_pixel(c)[0])
        axsRM[i].set_ylim(WCS(hdrRM).world_to_pixel(c)[1])

        lonFD = axsFD[i].coords[0]
        latFD = axsFD[i].coords[1]

        lonRM = axsRM[i].coords[0]
        latRM = axsRM[i].coords[1]

        lonFD.set_ticklabel_visible(False)

        axsFD[i].tick_params(axis='both', labelsize=fs)
        axsRM[i].tick_params(axis='both', labelsize=fs)
        
        lonFD.set_major_formatter('d.d')
        latFD.set_major_formatter('d.d')
        lonRM.set_major_formatter('d.d')
        latRM.set_major_formatter('d.d')
        
        lonFD.set_ticks(spacing=lonspace_list[i]* u.deg)  # Minor ticks every 1 degree
        latFD.set_ticks(spacing=latspace_list[i]* u.deg)   
        lonRM.set_ticks(spacing=lonspace_list[i]* u.deg)  # Minor ticks every 1 degree
        latRM.set_ticks(spacing=latspace_list[i]* u.deg)  
        
        axsFD[i].set_ylabel('  ',fontsize=fs)
        axsFD[i].set_xlabel('  ',fontsize=fs)
        axsFD[i].tick_params(axis='both', which='both', width=2, length=6)
        for spine in axsFD[i].spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)

        axsRM[i].set_ylabel('  ',fontsize=fs)
        axsRM[i].set_xlabel('  ',fontsize=fs)
        axsRM[i].tick_params(axis='both', which='both', width=2, length=6)
        for spine in axsRM[i].spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)

    for cbar in [cbarFD,cbarRM]:
        cbar.ax.tick_params(axis='y', which='both', width=2, length=6)
        cbar.ax.tick_params(labelsize=fs)
        cbar.outline.set_linewidth(2)

    
    #plt.savefig('../plots/maps/'+filename+'.png')

    return

In [ ]:
def FD_moments(data,peakthresh,maxFD,RM_arr,M0=True,M1=True,M2=False,*args,**kwargs):
 
    dFD = abs(RM_arr[1]-RM_arr[0])

    # cut out FD range chosen:
    data_use = data[(abs(RM_arr) <= maxFD),:,:]    
    RM_arr_use = RM_arr[(abs(RM_arr) <= maxFD)]  

    # set any data points below PI threshold to NaN:
    data_use[data_use < peakthresh] = np.nan

    moments = {}
    if M0:
        M0_data = dFD*np.nansum(data_use,axis=0)
        M0_data[M0_data == 0] = np.nan
        moments['M0'] = M0_data
    if M1:
        M1_data = dFD*np.nansum(data_use*RM_arr_use[:,np.newaxis,np.newaxis],axis=0)/M0_data
        moments['M1'] = M1_data
    if M2:
        M2_data = np.sqrt(dFD*np.nansum(data_use*(RM_arr_use[:,np.newaxis,np.newaxis]-M1_data)**2,axis=0)/M0_data)
        moments['M2'] = M2_data
        
    return(moments)

In [ ]:
FD_hdu = fits.open('/srv/data/cgps-gmims/gmims_FD/phi_peak_regrd.fits')

FD_G = FD_hdu[0].data
hdrFD = FD_hdu[0].header
#print(repr(hdrFD))
print(FD_G.shape)

In [ ]:
make_the_plots_v2(FD_G, RM_G_filt, hdrFD, hdr, [189.5,157.5,172.5], [3,0.0,-0.5],
                  [1,1,2],[0.5,0.5,1], [4.2,4.2,8.2], filename='gmims_FD_RM',
                  FDmax=50, RMmax=100, src=None)


In [ ]:
c = SkyCoord([158.9,158.5], [0.65,1.025], frame=Galactic, unit="deg")
fs = 24
fig = plt.figure(figsize=(29,8))
plt.subplots_adjust(wspace=0.1, left=0.05, right=0.85, top=0.98, bottom=0.1)

lonspace=0.1
latspace=0.1

cmap = mpl.colormaps.get_cmap('RdBu_r') 
cmap.set_bad(color='grey')

ax1  = fig.add_subplot(131, projection=WCS(hdr).celestial)
ax2  = fig.add_subplot(132, projection=WCS(hdr).celestial)
ax3  = fig.add_subplot(133, projection=WCS(hdr).celestial)

im1  = ax1.imshow(PA_CG*180/np.pi, origin='lower', vmin=-90, vmax=90, cmap='twilight')
im2  = ax2.imshow(RM_C_filt, origin='lower', vmin=-200, vmax=200, cmap=cmap)
im3  = ax3.imshow(RM_CG_filt, origin='lower', vmin=-200, vmax=200, cmap=cmap)

cbax = fig.add_axes([0.9, 0.1, 0.02, 0.88])
cbar = fig.colorbar(im3, cax=cbax, orientation='vertical')
cbar.set_label(r'RM (rad m$^{-2}$)', fontsize=fs)
cbar.ax.tick_params(axis='y', which='both', width=2, length=6)
cbar.ax.tick_params(labelsize=fs)
cbar.outline.set_linewidth(2)

for ax in [ax1,ax2,ax3]:
    ax.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    lon = ax.coords[0]
    lat = ax.coords[1]
    ax.tick_params(axis='both', labelsize=fs)
    lon.set_major_formatter('d.d')
    lat.set_major_formatter('d.d')
    lon.set_ticks(spacing=lonspace* u.deg)  # Minor ticks every 1 degree
    lat.set_ticks(spacing=latspace* u.deg)    
    ax.set_ylabel('  ',fontsize=fs)
    ax.set_xlabel('  ',fontsize=fs)
    ax.tick_params(axis='both', which='both', width=2, length=6)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)
    #ax.contour(PI[2], levels=contours, colors='black')

#plt.savefig('../plots/sh216_RM.png')

In [ ]:
# CGPS
hdu = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_C_conv4_regrd.fits')
rm_cgps  = hdu[0].data
hdr_cgps = hdu[0].header
print(rm_cgps.shape)

#hdu = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_C_conv4_regrd_PI_of_mean.fits')
#pi_cgps  = hdu[0].data

# CGPS+GMIMS
hdu = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_CG_conv4_regrd.fits')
rm_cgps_gmims  = hdu[0].data
hdr_cgps_gmims = hdu[0].header
print(rm_cgps_gmims.shape)

# Hutschenreuter
hdu = fits.open('/srv/aordog/faraday2020v2.fits')
rm_foreground_hpx  = hdu[1].data.faraday_sky_mean
hdr_foreground = hdu[1].header
print(rm_foreground_hpx.shape)
#print(repr(hdr_foreground))

# GMIMS-HBN
hdu = fits.open('/srv/data/gmims/gmims-hbn/GMIMS-HBN_v1_gal_car_FD_PI.fits')
gmims_hbn     = hdu[0].data
hdr_gmims_hbn = hdu[0].header
print(gmims_hbn.shape)

# DRAGONS
hdu = fits.open('/srv/data/dragons/Mar2024_500_1000_MHz_cube/FD_cube_DRAGONS_CLEAN_32.fits')
dragons     = hdu[0].data
hdr_dragons = hdu[0].header
print(dragons.shape)

# CHIME
hdu = fits.open('/srv/data/chime/chime_FD_May2024_400_729/FDF_clean_tot.fits')
chime     = hdu[0].data
hdr_chime = hdu[0].header
print(chime.shape)

## Make 2D headers

In [ ]:
wcs = WCS(hdr_chime)
hdr2D_chime = wcs.dropaxis(2).to_header()
hdr2D_chime['NAXIS'] = 2
hdr2D_chime['NAXIS1'] = chime.shape[2]
hdr2D_chime['NAXIS2'] = chime.shape[1]
print(repr(hdr2D_chime))

## Hutschenreuter to plate caree

In [ ]:
rm_foreground, footprint = reproject_from_healpix((rm_foreground_hpx, 'galactic'),hdr2D_chime, nested=False)

## Get FD axes

In [ ]:
FD_ax_chime = WCS(hdr_chime).all_pix2world(0,0,range(chime.shape[0]),0)[2]
FD_ax_dragons = WCS(hdr_dragons).all_pix2world(0,0,range(dragons.shape[0]),0)[2]
FD_ax_gmims_hbn = WCS(hdr_gmims_hbn).all_pix2world(0,0,range(gmims_hbn.shape[0]),0)[2]

## Calculate moments

In [ ]:
moments = FD_moments(gmims_hbn,0.04,450,FD_ax_gmims_hbn,M0=True,M1=True,M2=False)
m0_gmims_hbn = moments['M0']
m1_gmims_hbn = moments['M1']

moments = FD_moments(dragons,0.5,100,FD_ax_dragons,M0=True,M1=True,M2=False)
m0_dragons = moments['M0']
m1_dragons = moments['M1']

moments = FD_moments(chime,0.08,100,FD_ax_chime,M0=True,M1=True,M2=False)
m0_chime = moments['M0']
m1_chime = moments['M1']

In [ ]:
rm_data = [rm_cgps,       rm_cgps_gmims,       rm_foreground]
rm_wcss = [WCS(hdr_cgps), WCS(hdr_cgps_gmims), WCS(hdr2D_chime)]

fd_data = [m0_chime,         m0_gmims_hbn,                   m0_dragons]
fd_wcss = [WCS(hdr2D_chime), WCS(hdr_gmims_hbn).dropaxis(2), WCS(hdr_dragons).dropaxis(2)]

In [ ]:
fig = plt.figure(figsize=(20,15))

axRM1  = fig.add_subplot(231, projection=rm_wcss[0].celestial)
axRM2  = fig.add_subplot(232, projection=rm_wcss[1].celestial)
axRM3  = fig.add_subplot(233, projection=rm_wcss[2].celestial)

axFD1  = fig.add_subplot(234, projection=fd_wcss[0].celestial)
axFD2  = fig.add_subplot(235, projection=fd_wcss[1].celestial)
axFD3  = fig.add_subplot(236, projection=fd_wcss[2].celestial)

#c = SkyCoord([160,154], [-2,2], frame=Galactic, unit="deg")
c = SkyCoord([176,170], [-5,5], frame=Galactic, unit="deg")

axsRM = [axRM1,axRM2,axRM3]
axsFD = [axFD1,axFD2,axFD3]

vmaxRMs = [150,150,100]
#vmaxFDs = [15, 15, 15]
vmaxFDs = [50, 60, 200]

for i in range(0,3):

    cmap = mpl.colormaps.get_cmap('RdBu_r') 
    cmap.set_bad(color='grey')

    im  = axsRM[i].imshow(rm_data[i], origin='lower', vmin=-vmaxRMs[i], vmax=vmaxRMs[i], cmap=cmap)
    axsRM[i].set_xlim(rm_wcss[i].world_to_pixel(c)[0])
    axsRM[i].set_ylim(rm_wcss[i].world_to_pixel(c)[1])

    cmap = mpl.colormaps.get_cmap('viridis') 
    cmap.set_bad(color='grey')

    im  = axsFD[i].imshow(fd_data[i], origin='lower', vmin=0, vmax=vmaxFDs[i], cmap=cmap)
    axsFD[i].set_xlim(fd_wcss[i].world_to_pixel(c)[0])
    axsFD[i].set_ylim(fd_wcss[i].world_to_pixel(c)[1])

#plt.savefig('../plots/all_FD_RM.png')